In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from matplotlib.colors import ListedColormap
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import plotly.subplots as sp
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import time


# from google.colab import files
# uploaded = files.upload()  # Opens a file upload dialog


<h2>Loading and Exploring the Dataset</h2>

In [ ]:
df = pd.read_csv('data_banknote_authentication.csv')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

### There is no nulls to handle

In [ ]:
df.duplicated().sum()

There is 24 duplicates A small number so we can drop them & it will not affect the model

In [ ]:
df = df.drop_duplicates()

In [ ]:
# Split features and labels
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [ ]:
colors = ['#767497', '#f7c3d4']

for label in np.unique(y):
    plt.scatter(
        X[y == label, 0],
        X[y == label, 1],
        c=colors[label],
        label=f'Class {label}',
        edgecolors='k'
    )

plt.xlabel('Variance')
plt.ylabel('Skewness')
plt.title('Variance vs. Skewness')
plt.legend()
plt.show()

<h3>Dataset Overview</h3>
The dataset contains features extracted from banknote images using wavelet transforms.<br>
These features include variance, skewness, curtosis, and entropy. Each row is labeled as either a forged note (class 0) or a real note (class 1).<br>
This helps us understand the structure of the data before building models.<br>
Since the relationships between features and classes are not perfectly linear, this is a good reason to use a non-linear SVM,<br>
which can capture complex patterns better than a straight-line model.

This dataset contains **5 columns**:

- **variance**: Measures the variation in pixel intensity values.
- **skewness**: Describes asymmetry in the wavelet-transformed image.
- **curtosis**: Describes the sharpness or flatness of the distribution.
- **entropy** : Measures the randomness in the image.
- **class**   : Target label — `0` for forged banknotes, `1` for real banknotes.

In [ ]:
plt.figure(figsize=(12,6))
for i, col in enumerate(df.columns[:-1]):
    plt.subplot(2, 2, i+1)
    sns.boxplot(x='class', y=col, data=df, palette=colors)
    plt.title(f'Distribution of {col} by Class')
plt.tight_layout()
plt.show()

In [ ]:
# def remove_outliers_iqr(df, features):
#     df_clean = df.copy()
#     total_removed = 0

#     for col in features:
#         Q1 = df_clean[col].quantile(0.25)
#         Q3 = df_clean[col].quantile(0.75)
#         IQR = Q3 - Q1
#         lower = Q1 - 1.5 * IQR
#         upper = Q3 + 1.5 * IQR

#         before = df_clean.shape[0]
#         df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
#         removed = before - df_clean.shape[0]
#         total_removed += removed
#         print(f"Removed {removed} outliers from '{col}'")

#     print(f"Total outliers removed: {total_removed}")
#     print(f"New shape: {df_clean.shape}")
#     return df_clean

In [ ]:
# #Apply to numeric columns only
# numeric_features = df.select_dtypes(include=np.number).columns.tolist()
# numeric_features.remove('class')  # exclude the label

# df= remove_outliers_iqr(df, numeric_features)

🧮 **Variance**<br>
The variance feature shows a strong difference between the two classes. Most forged notes (class 0) have higher variance values,<br>
while real notes (class 1) have lower ones. This makes variance helpful for separating the classes. <br>
However, since there’s still a bit of overlap, a non-linear SVM would help capture the boundary more accurately than a straight line.


📉 **Skewness**<br>
Skewness has values that are mixed between the two classes. Forged notes tend to have more positive skewness,<br>
while real notes are closer to zero or negative. But the overlap is too large for a linear boundary to work well.<br>
A non-linear SVM can draw a curved boundary that fits the data better.

🎯 **Curtosis**<br>
Curtosis values are more spread out for real notes (class 1). This means real notes can have very different shapes in their data,<br>
while forged notes are more consistent. Because of this wide variation and overlap, a non-linear SVM is better at learning these complex patterns.


🔀 **Entropy**<br>
Entropy values look very similar for both classes. There are some small differences,<br>
like more low values in class 0, but overall it's hard to separate them using just a straight line.


In [ ]:
sns.pairplot(data=df, hue='class', palette=colors)
plt.suptitle('Pairwise Feature Relationships', y=1.02)
plt.show()

### 🔍 **Feature Relationships: Pairplot Visualization**<br>

This pairplot shows how the four features — **variance**, **skewness**, **curtosis**, and **entropy** — relate to each other,<br>
with points colored by class (0 = forged, 1 = real).

- **Scatter plots** (off-diagonal) show how pairs of features are distributed together.
- **Diagonal plots** show the **distribution (like a histogram or curve)** of each feature separately for each class.

📊 The **curves on the diagonal** are similar to **normal (bell-shaped) distributions**, but they are not perfectly symmetrical.  
This tells us that the feature values are **spread out differently** for real and forged notes. For example:
- Some features have values that are more concentrated or wider for one class.
- The curves often **overlap**, meaning the classes are not easily separable by just looking at one feature.

<h4>Since the data is not cleanly separated and shows complex patterns, it's better to use a non-linear SVM.<br>  
A non-linear model can **learn curved or flexible boundaries** to better divide the two classes.</h4>


In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

for label in np.unique(y):
    ax.scatter(
        X[y == label, 0],  # variance
        X[y == label, 1],  # skewness
        X[y == label, 2],  # curtosis
        c=colors[label],
        label=f'Class {label}',
        depthshade=True
    )

ax.set_xlabel('Variance')
ax.set_ylabel('Skewness')
ax.set_zlabel('Curtosis')
ax.set_title('3D View of Banknote Features')
plt.legend()
plt.show()

In [ ]:
# 2D PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8,6))
for label in np.unique(y):
    plt.scatter(
        X_pca[y == label, 0],
        X_pca[y == label, 1],
        c=colors[label],
        label=f'Class {label}',
        edgecolors='k'
    )
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('2D PCA Projection')
plt.legend()
plt.show()

# 3D PCA
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
for label in np.unique(y):
    ax.scatter(
        X_pca[y == label, 0],
        X_pca[y == label, 1],
        X_pca[y == label, 2],
        c=colors[label],
        label=f'Class {label}',
        depthshade=True
    )
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('3D PCA Projection')
plt.legend()
plt.show()

In [ ]:
# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

<h2>Grid Search for SVM Hyperparameters</h2>

In [ ]:
# define the parameter grid
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
              'gamma':[0.0001, 0.001, 0.01, 1, 10, 100, 1000]}

# perform grid search
svm = SVC(kernel='rbf')
grid_search = GridSearchCV(svm,
                           param_grid,
                           cv=3,
                           n_jobs=-1)
grid_search.fit(X_scaled, y)

print(
    "Best parameters are {} \nScore : {}%".format(
        grid_search.best_params_, grid_search.best_score_*100)
)

# Reshape for heatmap
scores = grid_search.cv_results_["mean_test_score"].reshape(
    len(param_grid['gamma']),
    len(param_grid['C']))

# Heatmap
sns.heatmap(scores,
            cmap = plt.cm.pink,
            annot= True,
            cbar= True,
            square=True)

plt.xlabel("gamma")
plt.ylabel("C")
plt.xticks(np.arange(len(param_grid['gamma'])), param_grid['gamma'], rotation=45)
plt.yticks(np.arange(len(param_grid['C'])), param_grid['C'], rotation=0)

plt.title("Accuracy for different parameters")
plt.show()

## Plot accuracy vs C parameter
plt.figure(figsize=(10, 6))
plt.title("Accuracy vs C parameter")
plt.xlabel("C")
plt.ylabel("Accuracy")
n = len(param_grid['C'])
for i in range(n):
    plt.plot(param_grid['C'],
             scores[:,i],
             'o-', label='gamma='+str(param_grid['gamma'][i]))

plt.legend()
plt.xscale('log')
plt.show()

<h2>Primal Support Vector Machine (SVM)<h2>

The SVM class implements a linear Support Vector Machine (SVM) using the primal form and batch subgradient descent to classify data into two categories (binary classification).<br>

Learns a hyperplane to separate two classes.

Minimizes hinge loss + L2 regularization.

Predicts new data using the learned model.

Tracks training loss and accuracy.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3,random_state=42)

In [ ]:
class SVM:
    def __init__(self, learning_rate=1e-3, lambda_param=1e-2, n_iters=300):
        """
        Initialize the Support Vector Machine (SVM) with the following hyperparameters:

        learning_rate (lr): The step size for gradient updates (default is 1e-3).
        lambda_param (λ): Regularization parameter that controls the trade-off between
                          fitting the margin and overfitting (default is 1e-2).
        n_iters (n_iters): Number of training iterations (epochs) for gradient descent (default is 300).
        """
        self.lr = learning_rate                  # Learning rate (η)
        self.lambda_param = lambda_param          # Regularization parameter (λ)
        self.n_iters = n_iters                    # Number of iterations (epochs)
        self.w = None                             # Weight vector (w), initially None
        self.b = None                             # Bias term (b), initially None
        self.losses = []                          # List to store loss at each epoch
        self.accuracies = []                      # List to store accuracy at each epoch

    def _init_weights_bias(self, X):
        """
        Initialize the weight vector (w) and bias term (b) to zero.

        The weight vector `w` has the same dimension as the number of features in X.
        The bias term `b` is a scalar and initialized to 0.
        """
        n_features = X.shape[1]                  # Get number of features from the input X
        self.w = np.zeros(n_features)            # Initialize weight vector to zeros: w ∈ ℝⁿ
        self.b = 0                                # Initialize bias term to zero (b = 0)

    def _decision_function(self, X):
      return np.dot(X, self.w) + self.b #f(x)=w⋅x+b


    def _get_cls_map(self, y):
        """
        Map the class labels from {0, 1} to {-1, 1} for the hinge loss.

        - Hinge loss is designed to work with labels in {-1, 1}.
        - This function converts binary class labels from {0, 1} to {-1, 1}.
        """
        return np.where(y <= 0, -1, 1)             # Convert 0 -> -1, 1 -> 1

    def _compute_loss(self, X, y):
        """
        Compute the total loss: regularized hinge loss.

        The loss consists of:
        1. Hinge loss: L_hinge = (1/m) * Σ max(0, 1 - yᵢ * (w⋅xᵢ + b))
        2. Regularization loss: L_reg = (λ/2) * ||w||²

        The final total loss is the sum of the hinge loss and regularization loss:
        L = L_hinge + L_reg
        """
        distances = 1 - y * (np.dot(X, self.w) + self.b)  # Calculate the margin for each data point
        distances = np.maximum(0, distances)             # Hinge loss: max(0, 1 - yᵢ(w⋅xᵢ + b))
        hinge_loss = np.mean(distances)                  # Mean hinge loss across all samples
        reg_loss = 0.5 * self.lambda_param * np.dot(self.w, self.w)  # regularization: (λ/2)‖w‖²
        return reg_loss + hinge_loss                     # Total loss = regularization + hinge loss

    def select_support_vectors(self, X, y):
        """
        Return the support vectors that lie on or inside the margin.

        Support vectors are defined as those points where:
        yᵢ(w⋅xᵢ + b) ≤ 1
        These are the points that influence the position of the decision boundary.
        """
        decision = y * (np.dot(X, self.w) + self.b)    # Calculate decision function for each point
        return X[decision <= 1 + 1e-5]                  # Support vectors are those satisfying this condition

    def fit(self, X, y):
        """
        Train the SVM using batch subgradient descent.

        The goal is to minimize the total loss (regularized hinge loss).
        For each sample, we compute gradients and update the weights and bias.

        Main components:
        - Hinge loss: penalizes samples that are inside or misclassified.
        - Regularization term: helps to avoid overfitting by penalizing large weights.
        """
        self._init_weights_bias(X)  # Initialize weights and bias
        y_original = y.copy()
        y = self._get_cls_map(y)    # Map labels to {-1, +1} for hinge loss formulation

        # Sub-Gradient descent for `n_iters`(epochs)
        for epoch in range(self.n_iters):
            # Determine which samples violate the margin (those inside the margin or misclassified)
            condition = y * (np.dot(X, self.w) + self.b) < 1

            # Compute gradients:
            # dw = λw - (1/n) * Σ yᵢ xᵢ for samples that violate the margin
            dw = self.lambda_param * self.w - np.dot(X[condition].T, y[condition]) / len(X)
            # db = - (1/n) * Σ yᵢ for samples that violate the margin
            db = -np.mean(y[condition])

            # Update weights and bias using the computed gradients and learning rate
            self.w -= self.lr * dw     # Update weight vector using the gradient
            self.b -= self.lr * db     # Update bias term using the gradient

            # Track metrics (accuracy and loss) to evaluate model performance during training
            acc = self.calculate_accuracy(X, y_original)
            loss = self._compute_loss(X, y)  # Compute the total loss (hinge + regularization)

            # Append loss and accuracy to the lists for later plotting
            self.losses.append(loss)
            self.accuracies.append(acc)

            # Print the current epoch, loss, and accuracy for monitoring
            # print(f"Epoch {epoch+1}/{self.n_iters} - Loss: {loss:.4f}, Accuracy: {acc:.4f}")

    def predict(self, X):
        """
        Predict the class labels for the input data X.

        The decision function is given by f(x) = w⋅x + b. The prediction is:
        - If f(x) ≥ 0, predict Class 1 (positive class)
        - If f(x) < 0, predict Class 0 (negative class)
        """
        linear_output = np.dot(X, self.w) + self.b   # Compute the decision function for each sample
        return np.where(linear_output >= 0, 1, 0)    # Return 1 if f(x) ≥ 0, else 0 (Class 1 or Class 0)

    def calculate_accuracy(self, X, y_true):
        y_pred = self.predict(X)
        return accuracy_score(y_true, y_pred) * 100


**Visualizing Training Progress**<br>
The plot_results function is used to visualize the training performance of the custom SVM classifier over time. It generates an interactive line plot using Plotly to display:

Accuracy over epochs (how well the model is learning to classify correctly)<br>
Loss over epochs (how well the model is minimizing error during training)

In [ ]:
def plot_results(svm):
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        y=svm.accuracies,
        mode='lines',
        name='Accuracy',
        line=dict(color='#767497')
    ))
    fig2.add_trace(go.Scatter(
        y=svm.losses,
        mode='lines',
        name='Loss',
        line=dict(color='#f7c3d4')
    ))
    fig2.update_layout(
        title='Accuracy and Loss over Epochs',
        xaxis_title='Epoch',
        yaxis_title='Value'
    )
    fig2.show()



### Visualizing the SVM Decision Boundary in 2D:

This function visualizes how the SVM separates two classes using the first two features.  
It shows the decision boundary, margins, and highlights support vectors.  
Useful for understanding how well the SVM model classifies in 2D space.  
Set `dual=True` if using the dual SVM model.  


In [ ]:
def plot_2d(svm, dual=False):
    X_2d = X_scaled[:, :2]
    svm_2d = svm
    svm_2d.fit(X_2d, y)
    y_pred_2d = svm_2d.predict(X_2d)

    # Create grid for contour
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.linspace(x_min, x_max, 500), np.linspace(y_min, y_max, 500)
    xx_grid, yy_grid = np.meshgrid(xx, yy)
    grid = np.c_[xx_grid.ravel(), yy_grid.ravel()]

    # Compute decision values
    if dual:
        Z = svm_2d._decision_function_batch(grid).reshape(xx_grid.shape)
        b = svm_2d.b
        w = None
    else:
        Z = svm_2d._decision_function(grid).reshape(xx_grid.shape)
        w = svm_2d.w
        b = svm_2d.b

    def compute_line(w, b, margin, x_vals):
        return (-w[0]*x_vals - b + margin) / w[1]

    fig = go.Figure()

    # Background classification regions
    fig.add_trace(go.Contour(
        x=xx, y=yy, z=Z,
        showscale=False,
        colorscale=[[0, '#f7c3d4'], [1, '#767497']],
        contours=dict(
            start=-2, end=2, size=0.5,
            coloring='fill',
            showlines=False
        ),
        line_smoothing=0.85,
        opacity=0.6,
        name='Decision Surface'
    ))

    # Decision boundary (Z=0) in black
    fig.add_trace(go.Contour(
        x=xx, y=yy, z=Z,
        contours=dict(
            start=0, end=0, size=1,
            coloring='lines',
            showlines=True
        ),
        line=dict(color='green', width=1),
        colorscale=[[0, 'green'], [1, 'green']],
        showscale=False,
        hoverinfo='skip',
        name='Decision Boundary'
    ))

    # Margins (only in primal)
    if w is not None:
        x_line = np.linspace(x_min, x_max, 500)
        y_decision = compute_line(w, b, 0, x_line)
        y_margin_pos = compute_line(w, b, 1, x_line)
        y_margin_neg = compute_line(w, b, -1, x_line)

        fig.add_trace(go.Scatter(x=x_line, y=y_margin_pos,
            mode='lines', line=dict(color='green', width=1, dash='dot'),
            name='Margin +1'
        ))
        fig.add_trace(go.Scatter(x=x_line, y=y_margin_neg,
            mode='lines', line=dict(color='green', width=1, dash='dot'),
            name='Margin -1'
        ))

    # Data points
    fig.add_trace(go.Scatter(
        x=X_2d[:, 0], y=X_2d[:, 1],
        mode='markers',
        marker=dict(
            color=['#f7c3d4' if label == -1 else '#767497' for label in y],
            size=6, line=dict(width=0.5, color='black')
        ),
        name='Data Points',
        text=[f"True: {yt}, Pred: {yp}" for yt, yp in zip(y, y_pred_2d)],
        hoverinfo='text'
    ))

    # Support vectors
    if dual:
        alpha = svm_2d.alpha
        idx, sv, sv_y, sv_alpha = svm_2d.select_support_vectors(alpha, X_2d, svm_2d.y_train)
        support_vecs = sv
        support_vals = svm_2d._decision_function_batch(support_vecs)
        # Filter out margin vectors
        on_margin_mask = np.isclose(np.abs(support_vals), 1, atol=1e-2)
        support_vecs = support_vecs[~on_margin_mask]
    else:
        y_cls = svm_2d._get_cls_map(y)
        support_vecs = svm_2d.select_support_vectors(X_2d, y_cls)
        support_vals = svm_2d._decision_function(support_vecs)
        on_margin_mask = np.isclose(np.abs(support_vals), 1, atol=1e-2)
        margin_vecs = support_vecs[on_margin_mask]
        non_margin_vecs = support_vecs[~on_margin_mask]

        fig.add_trace(go.Scatter(
            x=margin_vecs[:, 0], y=margin_vecs[:, 1],
            mode='markers', marker=dict(color='red', size=10, symbol='x'),
            name='Support Vectors on Margin'
        ))

        fig.add_trace(go.Scatter(
            x=non_margin_vecs[:, 0], y=non_margin_vecs[:, 1],
            mode='markers',
            marker=dict(color='rgba(255,100,100,0.3)', size=9, line=dict(width=1, color='darkred')),
            name='Other Support Vectors'
        ))

    # Layout
    f1, f2 = df.columns[0], df.columns[1]
    fig.update_layout(
        title='2D SVM Decision Boundary',
        xaxis_title=f1,
        yaxis_title=f2,
        showlegend=True,
        yaxis=dict(range=[-3, 2.5])
    )

    fig.show()


### Visualizing the SVM Decision Boundary in 3D:

This function visualizes the SVM classification results in 3D using the first three features.  
It plots data points colored by predicted class and shows true vs. predicted labels on hover.  
Useful for understanding SVM behavior in higher dimensions.  
Supports binary classification with label sets {-1,1} or {0,1}.

In [ ]:
def plot_3d(svm):
    X_3d = X_scaled[:, :3]
    svm_3d = svm
    svm_3d.fit(X_3d, y)
    y_pred_3d = svm_3d.predict(X_3d)
    hover_text_3d = [f"True: {yt}, Pred: {yp}" for yt, yp in zip(y, y_pred_3d)]

    # Normalize predictions to {0, 1}
    unique_labels = np.unique(y_pred_3d)
    if set(unique_labels) == {-1, 1}:
        colors = ['#f7c3d4' if c == -1 else '#767497' for c in y_pred_3d]
    elif set(unique_labels) == {0, 1}:
        colors = ['#f7c3d4' if c == 0 else '#767497' for c in y_pred_3d]
    else:
        raise ValueError(f"Unexpected class labels: {unique_labels}")

    fig_3d = go.Figure(data=[go.Scatter3d(
        x=X_3d[:, 0],
        y=X_3d[:, 1],
        z=X_3d[:, 2],
        mode='markers',
        marker=dict(
            size=5,
            color=colors,
            opacity=0.7
        ),
        text=hover_text_3d,
        hoverinfo='text',
        name='3D Points'
    )])

    f1, f2, f3 = df.columns[:3]
    fig_3d.update_layout(
        title="3D SVM Visualization",
        scene=dict(
            xaxis_title=f1,
            yaxis_title=f2,
            zaxis_title=f3
        )
    )

    fig_3d.show()


### cross-validation for an SVM model
This function performs k-fold cross-validation on an SVM model, testing different combinations of learning rates and regularization parameters.<br>
It evaluates each model's performance using a combination of accuracy and loss, selecting the hyperparameters that maximize the combined score.

In [ ]:
def cross_validate_primal(X, y, learning_rates, lambda_params, k=5, n_iters=1000, alpha=0.5):
    n_samples = len(X)
    fold_size = n_samples // k
    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    best_params = None
    best_score = -np.inf

    for lr in learning_rates:
        for lam in lambda_params:
            combined_scores = []

            for fold in range(k):
                # Split manually
                val_idx = indices[fold * fold_size: (fold + 1) * fold_size]
                train_idx = np.setdiff1d(indices, val_idx)

                X_train, y_train = X[train_idx], y[train_idx]
                X_val, y_val = X[val_idx], y[val_idx]

                # Train model
                model = SVM(learning_rate=lr, lambda_param=lam, n_iters=n_iters)
                model.fit(X_train, y_train)

                # Predict on validation
                preds = model.predict(X_val)
                y_val_binary = np.where(y_val <= 0, 0, 1)
                acc = np.mean(preds == y_val_binary)

                # Compute loss on validation
                y_val_hinge = model._get_cls_map(y_val)
                val_loss = model._compute_loss(X_val, y_val_hinge)

                # Combine score: higher accuracy, lower loss
                score = acc - alpha * val_loss
                combined_scores.append(score)

            avg_score = np.mean(combined_scores)
            print(f"lr={lr}, lambda={lam} => combined score: {avg_score:.4f}")

            if avg_score > best_score:
                best_score = avg_score
                best_params = (lr, lam)

    print(f"\nBest params (acc-loss combo): lr={best_params[0]}, lambda={best_params[1]}")
    return best_params


In [ ]:
# learning_rates = [0.001, 0.01, 0.1]
# lambda_params = [0.0001, 0.001,0.01 ,0.1, 1]
# best_lr, best_lambda = cross_validate_primal(X_train, y_train, learning_rates, lambda_params, k=5, n_iters=300)

best_lr = 0.1
best_lambda = 0.0001

<h3>Primal SVM Training and Evaluation</h3>

In [ ]:
# Train the SVM
svm_primal = SVM(learning_rate = best_lr,  lambda_param = best_lambda, n_iters = 2000)
svm_primal.fit(X_train, y_train)

# Predict
y_pred = svm_primal.predict(X_test)

# Accuracy
acc = svm_primal.calculate_accuracy(X_test, y_test)
print(f"Accuracy on my data (Primal Hinge): {acc:.2f}%")

In [ ]:
# Plot learning curv
plot_results(svm_primal)


In [ ]:
model = SVM(learning_rate = best_lr,  lambda_param = best_lambda, n_iters = 2000)
plot_2d(model)

In [ ]:
plot_3d(model)

<h2>Dual SVM with RBF Kernel</h2>

The RBF kernel computes similarity between two points using e^(-γ||x₁-x₂||²)<br>
gamma controls the influence of each training example (higher gamma = more complex decision boundary)

### Parameters:<br>
C: Regularization parameter (trade-off between margin and classification error)<br>
lr: Learning rate for gradient updates<br>
epochs: Number of training iterations<br>
gamma: RBF kernel parameter

### Key Concepts:<br>
Dual Formulation: Solves the SVM problem by optimizing the dual variables (α) rather than the primal (w, b)

Kernel Trick: Allows working in high-dimensional feature space without explicit computation via the kernel matrix

Support Vectors: Only a subset of training points (those with α > 0) influence the decision boundary

In [ ]:
class dual_SVM_RBF:
    def __init__(self, C=1.0, lr=0.001, epochs=10000, gamma=10):
        self.C = C
        self.lr = lr
        self.epochs = epochs
        self.gamma = gamma
        self.alpha = None     # Dual coefficients
        self.b = 0
        self.losses = []                          # List to store loss at each epoch
        self.accuracies = []                      # List to store accuracy at each epoch

    def rbf_kernel(self, x1, x2):
        return np.exp(-self.gamma * np.linalg.norm(x1 - x2)**2)

    def compute_kernel_matrix(self, X):
        n_samples = X.shape[0]
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self.rbf_kernel(X[i], X[j])
        return K

    def select_support_vectors(self, alpha, X, y):
        idx = np.where((alpha > 1e-5) & (alpha < self.C))[0]
        return idx, X[idx], y[idx], alpha[idx]

    def compute_bias(self, X, y, alpha, K):
        support_idx, _, _, _ = self.select_support_vectors(alpha, X, y)
        if len(support_idx) == 0:
            return 0

        b_total = 0
        for i in support_idx:
            f_xi = np.mean(alpha * y * K[:, i])
            b_total += (y[i] - f_xi)

        return b_total / len(support_idx)

    def _decision_function(self, x):
        result = 0
        for alpha, sv_y, sv in zip(self.support_vectors_alpha, self.support_vector_labels, self.support_vectors):
            result += alpha * sv_y * self.rbf_kernel(x, sv)
        return result + self.b

    def _decision_function_batch(self, X):
        # X shape: (n_samples, n_features)
        X_norm = np.sum(X**2, axis=1).reshape(-1, 1)
        SV_norm = np.sum(self.support_vectors**2, axis=1).reshape(1, -1)
        K = np.exp(-self.gamma * (X_norm - 2 * X @ self.support_vectors.T + SV_norm))  # shape (n_samples, n_sv)

        # Compute decision values
        decision = K @ (self.support_vectors_alpha * self.support_vector_labels) + self.b  # shape (n_samples,)
        return decision


    def predict(self, X):
        y_pred = []
        for x in X:
            decision = self._decision_function(x)
            y_pred.append(np.sign(decision))
        return np.array(y_pred)

    def calculate_accuracy(self, X, y_true):
        y_pred = self.predict(X)
        correct = np.sum(y_true == y_pred)
        return (correct / len(y_true)) * 100

    def _obj_dual(self, y, K):
        # f = np.dot(K, self.alpha * y) + self.b
        # hinge_loss = np.sum(np.maximum(0, 1 - y * f))
        # reg_term = 0.5 * np.dot(self.alpha, np.dot(K, self.alpha * y) * y)
        # return reg_term + self.C * hinge_loss

        term1 = np.sum(self.alpha)
        term2 = 0.5 * np.sum(
            (self.alpha[:, None] * self.alpha[None, :]) *
            (y[:, None] * y[None, :]) * K
        )
        return (term1 - term2)

    def _calculate_loss(self,y,f):
        loss = np.mean(np.maximum(0, 1 - y * f))
        return loss


    def fit(self, X, y):
        n_samples = X.shape[0]
        self.X_train = X
        self.y_train = y

        alpha = np.zeros(n_samples)
        K = self.compute_kernel_matrix(X)

        for epoch in range(self.epochs):
                f = K @ (alpha * y) + self.b
                condition = y * f < 1
                gradient = np.where(condition, 1 - y * f, -y * f)
                alpha += self.lr * gradient
                alpha = np.clip(alpha, 0, self.C)
                self.b = self.compute_bias(X, y, alpha, K)



                if epoch %100 == 0  :
                    idx, sv, sv_labels, sv_alpha = self.select_support_vectors(alpha, X, y)
                    self.support_vectors = sv
                    self.support_vector_labels = sv_labels
                    self.support_vectors_alpha = sv_alpha
                    self.alpha = alpha
                    current_obj = self._obj_dual(y, K)
                    current_loss = self._calculate_loss(y,f)
                    current_acc = self.calculate_accuracy(X,y)
                    self.losses.append(current_loss)
                    self.accuracies.append(current_acc)
                    print(f"Epoch {epoch}/{self.epochs}, Dual objective: {current_obj:.4f} , Loss:{current_loss:.4f}, Accuracy: {current_acc:.2f}")


        self.alpha = alpha
        self.b = self.compute_bias(X, y, alpha, K)

        # Save support vectors
        idx, sv, sv_labels, sv_alpha = self.select_support_vectors(alpha, X, y)
        self.support_vectors = sv
        self.support_vector_labels = sv_labels
        self.support_vectors_alpha = sv_alpha

In [ ]:
# # Convert labels from {0, 1} to {-1, 1} for hinge loss
y = np.where(y == 0, -1, 1)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3,random_state=42)

### Dual SVM Hyperparameter Tuning via k-Fold Cross-Validation<br>
cross_validate_dual(X, y, learning_rates, C_params, gamma_params, k=5, epochs=1000, alpha=0.5)

This function performs k-fold cross-validation to find the best hyperparameters (lr, C, gamma) for the dual_SVM_RBF model by evaluating a combined score (balancing accuracy and loss).

In [ ]:
def cross_validate_dual(X, y, learning_rates, C_params, gamma_params, k=5, epochs=1000, alpha=0.5):
    n_samples = len(X)
    fold_size = n_samples // k
    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    best_params = None
    best_score = -np.inf

    for lr in learning_rates:
        for C in C_params:
            for gamma in gamma_params:
                combined_scores = []

                for fold in range(k):
                    val_idx = indices[fold * fold_size: (fold + 1) * fold_size]
                    train_idx = np.setdiff1d(indices, val_idx)

                    X_train, y_train = X[train_idx], y[train_idx]
                    X_val, y_val = X[val_idx], y[val_idx]

                    # Train model
                    model = dual_SVM_RBF(C=C, lr=lr, epochs=epochs, gamma=gamma)
                    model.fit(X_train, y_train)

                    # acc on validation
                    acc = model.calculate_accuracy(X_val, y_val)

                    # Decision function values for loss
                    f = np.array([model._decision_function(x) for x in X_val])
                    val_loss = model._calculate_loss(y_val, f)

                    # Combine score: higher accuracy, lower loss
                    score = acc - alpha * val_loss
                    combined_scores.append(score)

                avg_score = np.mean(combined_scores)
                print(f"lr={lr}, C={C}, gamma={gamma} => combined score: {avg_score:.4f}")

                if avg_score > best_score:
                    best_score = avg_score
                    best_params = (lr, C, gamma)

    print(f"\nBest params (acc-loss combo): lr={best_params[0]}, C={best_params[1]}, gamma={best_params[2]}")
    return best_params

In [ ]:
# learning_rates = [0.001, 0.01, 0.1]
# C_params = [0.001, 0.01, 0.1, 1, 10]
# gamma_params =[0.001, 0.01, 1, 10]

# best_params = cross_validate_dual(
#     X_train, y_train,
#     learning_rates=learning_rates,
#     C_params=C_params,
#     gamma_params=gamma_params,
#     k=5,
#     epochs=2000,
#     alpha=0.3
# )

# best_params

In [ ]:
# lr_best, C_best, gamma_best = best_params

# Best params : lr=0.001, C=1, gamma=10

lr_best =0.001
C_best = 1
gamma_best = 10

dual_svm = dual_SVM_RBF(C=C_best, lr=lr_best, epochs= 2000, gamma=gamma_best)
dual_svm.fit(X_train, y_train)

# Predict
y_pred = dual_svm.predict(X_test)

# Accuracy
acc = dual_svm.calculate_accuracy(X_test, y_test)
print(f"Accuracy on my data: {acc:.2f}%")

In [ ]:
# Plot learning curv
plot_results(dual_svm)

In [ ]:
model = dual_SVM_RBF(C=C_best, lr=lr_best, epochs= 300, gamma=gamma_best)
plot_2d(model, dual = True)

In [ ]:
plot_3d(model)

<h2>SVM with Squared Hinge Loss and Gradient Descent</h2>

This class implements a linear SVM classifier trained using batch gradient descent. It:
- Initializes model weights and hyperparameters.
- Uses squared hinge loss to penalize points inside the margin or misclassified.
- Computes gradients to update weights and bias over many iterations.
- Optionally prints the training and validation loss and accuracy.
- Stores training history for analysis or plotting.

In [ ]:
class SVM_Squared_Hinge_Loss:
    def __init__(self, lr=0.01, lambda_param=1e-2, n_iters=2000, verbose=True):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.verbose = verbose
        self.w = None
        self.b = None

        self.history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
        }

    def _decision_function(self, X):
        return np.dot(X, self.w) + self.b

    def _squared_hinge_loss(self, X, y):
        margin = y * self._decision_function(X)
        hinge_loss = np.maximum(0, 1 - margin) ** 2
        total_loss = np.mean(hinge_loss) + 0.5 * self.lambda_param * np.dot(self.w, self.w)

        grad_w = -2 * np.dot((y * np.maximum(0, 1 - margin)), X) / len(X) + self.lambda_param * self.w
        grad_b = -2 * np.sum(y * np.maximum(0, 1 - margin)) / len(X)

        return total_loss, grad_w, grad_b

    def _accuracy(self, y_true, y_pred):
        return np.mean(y_true == y_pred)

    def predict(self, X):
        return np.sign(self._decision_function(X))

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        n_features = X_train.shape[1]
        self.w = np.zeros(n_features)
        self.b = 0

        for epoch in range(self.n_iters):
            # Batch gradient update
            total_loss, grad_w, grad_b = self._squared_hinge_loss(X_train, y_train)
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            # Optional: weight clipping
            self.w = np.clip(self.w, -10, 10)

            # Metrics
            train_pred = self.predict(X_train)
            train_acc = self._accuracy(y_train, train_pred)
            self.history['train_loss'].append(total_loss)
            self.history['train_acc'].append(train_acc)

            if X_val is not None:
                val_loss, _, _ = self._squared_hinge_loss(X_val, y_val)
                val_pred = self.predict(X_val)
                val_acc = self._accuracy(y_val, val_pred)
                self.history['val_loss'].append(val_loss)
                self.history['val_acc'].append(val_acc)

            if self.verbose and (epoch % 10 == 0 or epoch == self.n_iters - 1):
                msg = f"Epoch {epoch+1}/{self.n_iters} | Train Loss: {total_loss:.4f} | Train Acc: {train_acc:.3f}"
                if X_val is not None:
                    msg += f" | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}"
                print(msg)

## Dataset Splitting and Model Training

- The dataset is split into 60% training, 20% validation, and 20% test sets.
- A custom SVM model is trained using gradient descent and squared hinge loss.
- The validation set monitors performance during training.
- After training, the model makes predictions on the test set to evaluate generalization.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

squared_hinge_svm = SVM_Squared_Hinge_Loss()
squared_hinge_svm.fit(X_train, y_train, X_val, y_val)

predictions = squared_hinge_svm.predict(X_test)

In [ ]:
y_pred = squared_hinge_svm.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

## Training and Validation Curves<br>

The plots below show how the SVM model's performance evolved over the training process:

- **Left Plot:** Training and validation loss over epochs.
  - Both losses decrease together with no significant gap, indicating the model is learning consistently and generalizing well.
  
- **Right Plot:** Training and validation accuracy over epochs.
  - Accuracy increases steadily, showing that the model is improving in its ability to classify correctly over time.

These curves indicate stable training with no overfitting, and good generalization to the validation set.


In [ ]:
train_loss = squared_hinge_svm.history['train_loss']
train_acc = squared_hinge_svm.history['train_acc']
val_loss = squared_hinge_svm.history['val_loss']
val_acc = squared_hinge_svm.history['val_acc']
epochs = list(range(1, len(train_loss) + 1))

# Create 1x2 subplot layout
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss over Epochs', 'Accuracy over Epochs'))

# --- Loss Plot ---
fig.add_trace(go.Scatter(
    x=epochs, y=train_loss, name='Train Loss',
    line=dict(color='blue')
), row=1, col=1)

if val_loss:
    fig.add_trace(go.Scatter(
        x=epochs, y=val_loss, name='Validation Loss',
        line=dict(color='orange')
    ), row=1, col=1)

# --- Accuracy Plot ---
fig.add_trace(go.Scatter(
    x=epochs, y=train_acc, name='Train Accuracy',
    line=dict(color='green')
), row=1, col=2)

if val_acc:
    fig.add_trace(go.Scatter(
        x=epochs, y=val_acc, name='Validation Accuracy',
        line=dict(color='red')
    ), row=1, col=2)

# Layout settings
fig.update_layout(
    title_text='Training and Validation Metrics over Epochs',
    xaxis_title='Epoch',
    xaxis2_title='Epoch',
    yaxis_title='Loss',
    yaxis2_title='Accuracy',
    legend_title='Legend',
    height=500,
    width=1000,
)

fig.show()

In [ ]:
def plot_loss_vs_accuracy(svm):

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        y=svm.history['train_acc'],
        mode='lines',
        name='Train Accuracy',
        line=dict(color='#767497')
    ))
    fig2.add_trace(go.Scatter(
        y=svm.history['train_loss'],
        mode='lines',
        name='Train Loss',
        line=dict(color='#f7c3d4')
    ))

    if svm.history['val_acc']:  
        fig2.add_trace(go.Scatter(
            y=svm.history['val_acc'],
            mode='lines',
            name='Val Accuracy',
            line=dict(color='#6fcf97')
        ))
        fig2.add_trace(go.Scatter(
            y=svm.history['val_loss'],
            mode='lines',
            name='Val Loss',
            line=dict(color='#f2994a')
        ))

    fig2.update_layout(
        title='Accuracy and Loss over Epochs',
        xaxis_title='Epoch',
        yaxis_title='Value'
    )
    fig2.show()

In [ ]:
plot_loss_vs_accuracy(squared_hinge_svm)

<h3>SVM Decision Boundary using Variance and Skewness</h3>

In [ ]:
def plot_decision_boundary(svm, X, y, feature_names=['variance', 'skewness']):

    # Get indices of features to plot
    feat_idx = [list(df.columns).index(f) for f in feature_names]
    X_2d = X[:, feat_idx]

    # Create grid space
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                         np.linspace(y_min, y_max, 500))

    # Create fake 4D grid with other features at mean values
    grid_4d = np.zeros((xx.ravel().shape[0], 4))
    grid_4d[:, feat_idx[0]] = xx.ravel()
    grid_4d[:, feat_idx[1]] = yy.ravel()
    for i in range(4):
        if i not in feat_idx:
            grid_4d[:, i] = np.mean(X[:, i])

    # Get decision values
    Z = svm._decision_function(grid_4d).reshape(xx.shape)
    y_pred = svm.predict(X)

    # Compute decision boundary and margins (using only the 2D weights)
    w_2d = svm.w[feat_idx]
    x_line = np.linspace(x_min, x_max, 500)
    y_decision = (-w_2d[0]*x_line - svm.b) / w_2d[1]
    y_margin_pos = (-w_2d[0]*x_line - svm.b + 1) / w_2d[1]
    y_margin_neg = (-w_2d[0]*x_line - svm.b - 1) / w_2d[1]

    # Approximate support vectors
    distances = np.abs(svm._decision_function(X))
    support_mask = distances <= 1.0 + 1e-2

    # Create plot
    plt.figure(figsize=(12, 8))

    # Background colors
    plt.contourf(xx, yy, Z, levels=[-100, 0, 100],
                 colors=['#f7c3d4', '#767497'], alpha=0.3)

    # Decision boundary and margins
    plt.plot(x_line, y_decision, 'k-', linewidth=2,
             label=f'Decision Boundary')
    plt.plot(x_line, y_margin_pos, 'k--', linewidth=1,
             label='Margin +1')
    plt.plot(x_line, y_margin_neg, 'k--', linewidth=1,
             label='Margin -1')

    # Data points (colored by true class)
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y,
                cmap=ListedColormap(['#ffb8cf', '#767497']),
                edgecolors='w', s=80, label='Data Points')

    # Support vectors
    plt.scatter(X_2d[support_mask, 0], X_2d[support_mask, 1],
                facecolors='none', edgecolors='red',
                s=80, linewidths=1,
                label='Support Vectors')

    # Formatting
    plt.xlabel(feature_names[0], fontsize=12)
    plt.ylabel(feature_names[1], fontsize=12)
    plt.title(f'SVM Decision Boundary', fontsize=14)

    # Legend
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_decision_boundary(squared_hinge_svm, X_train, y_train, feature_names=['variance', 'skewness'])

In [ ]:
model = SVM_Squared_Hinge_Loss()
plot_3d(model)

# **Accuracy Comparison**

In [ ]:
# --- Step 1: Recreate Data Splits for each model's original evaluation ---

# Original y {0, 1}
y_original_01 = df.iloc[:, -1].values
# y converted to {-1, 1}
y_neg11 = np.where(y_original_01 == 0, -1, 1)

# 1.1 Data for Primal SVM (Hinge Loss)
X_train_primal, X_test_primal, y_train_primal_01, y_test_primal_01 = train_test_split(
    X_scaled, y_original_01, test_size=0.3, random_state=42
)

# 1.2 Data for Dual SVM (RBF Kernel)
X_train_dual, X_test_dual, y_train_dual_neg11, y_test_dual_neg11 = train_test_split(
    X_scaled, y_neg11, test_size=0.3, random_state=42
)

# 1.3 Data for Squared Hinge Loss SVM
# Needs 60/20/20 split
X_train_sq_hinge_full, X_temp, y_train_sq_hinge_full, y_temp = train_test_split(
    X_scaled, y_neg11, test_size=0.4, random_state=42 # 60% train_full, 40% temp
)
X_val_sq_hinge, X_test_sq_hinge, y_val_sq_hinge, y_test_sq_hinge = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42 # Splitting temp 50/50 gives 20% val, 20% test
)
# The Squared Hinge fit method was called with X_train_sq_hinge_full, y_train_sq_hinge_full


# --- Step 2: Define Hyperparameters used for original training ---

# 2.1 Primal Hinge SVM
lr_primal = 0.1       
lambda_primal = 0.0001
n_iters_primal = 300  

# 2.2 Dual RBF SVM
lr_dual = 0.001      
C_dual = 1           
gamma_dual = 10       
epochs_dual = 300    

# 2.3 Squared Hinge SVM
lr_sq_hinge = 0.01      
lambda_sq_hinge = 1e-2
n_iters_sq_hinge = 2000 

# --- Step 3: Retrain models to ensure original state ---

print("Retraining models to match original evaluation state...")
start_time = time.time()

# 3.1 Retrain Primal SVM (Hinge Loss)
print("  Retraining Primal SVM (Hinge)...")
svm_primal_retrained = SVM(learning_rate=lr_primal, lambda_param=lambda_primal, n_iters=n_iters_primal)
# Suppress print statements during fit if needed (modify class or redirect stdout)
svm_primal_retrained.fit(X_train_primal, y_train_primal_01)

# 3.2 Retrain Dual SVM (RBF Kernel)
print("  Retraining Dual SVM (RBF)...")
model_dual_retrained = dual_SVM_RBF(C=C_dual, lr=lr_dual, epochs=epochs_dual, gamma=gamma_dual)
# Suppress print statements during fit if needed
model_dual_retrained.fit(X_train_dual, y_train_dual_neg11)

# 3.3 Retrain Squared Hinge SVM
print("  Retraining Squared Hinge SVM...")
svm_sq_hinge_retrained = SVM_Squared_Hinge_Loss(lr=lr_sq_hinge, lambda_param=lambda_sq_hinge, n_iters=n_iters_sq_hinge, verbose=False) # verbose=False to suppress output
svm_sq_hinge_retrained.fit(X_train_sq_hinge_full, y_train_sq_hinge_full, X_val_sq_hinge, y_val_sq_hinge) # Fit with train and val as done originally

end_time = time.time()
print(f"Retraining complete. Took {end_time - start_time:.2f} seconds.")


# --- Step 4: Calculate Accuracy using the specific test set for each ---

print("\nCalculating final accuracies...")
# 4.1 Accuracy for Primal SVM (Hinge)
acc_primal = svm_primal_retrained.calculate_accuracy(X_test_primal, y_test_primal_01)

# 4.2 Accuracy for Dual SVM (RBF)
acc_dual = model_dual_retrained.calculate_accuracy(X_test_dual, y_test_dual_neg11)

# 4.3 Accuracy for Squared Hinge SVM
y_pred_sq_hinge = svm_sq_hinge_retrained.predict(X_test_sq_hinge)
acc_sq_hinge = svm_sq_hinge_retrained._accuracy(y_test_sq_hinge, y_pred_sq_hinge) * 100


# --- Step 5: Print Results ---
print("\n------ Model Accuracy Comparison ------")
model_names = ['Primal SVM (Hinge)', 'Dual SVM (RBF)', 'SVM (Squared Hinge)']
accuracies = [acc_primal, acc_dual, acc_sq_hinge]

for name, acc in zip(model_names, accuracies):
    print(f"{name}: {acc:.2f}%")


# --- Step 6: Visualize Results ---
plt.figure(figsize=(10, 6))

# Choose distinct colors
colors = ['#767497', '#f7c3d4', '#aadd99'] # You can customize these
bars = plt.bar(model_names, accuracies, color=colors)

plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Comparison of Custom SVM Model Accuracies', fontsize=14)
# Adjust y-axis limits dynamically
min_acc = min(accuracies) if accuracies else 0
plt.ylim(max(0, min_acc - 5) , 105) # Start slightly below minimum accuracy but not below 0

# Add accuracy values on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval, f'{yval:.2f}%', va='bottom', ha='center', fontsize=10)

plt.xticks(rotation=10, fontsize=10) # Rotate labels slightly if needed
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7) # Add horizontal grid lines
plt.tight_layout() # Adjust layout
plt.show()

# **Convergence Speed**

In [ ]:
# --- Convergence Speed Comparison Cell ---

print("--- Comparing Convergence Speed ---")

# --- Access original model objects and their history ---
# !!! Assumes 'svm_primal', 'model', and 'svm' contain the originally trained objects !!!

model_histories = {}
model_names = []

# 1. Primal SVM (Hinge Loss)
try:
    # Check if the variable exists
    if 'svm_primal' in locals() or 'svm_primal' in globals():
        # Check if it has the expected attributes (basic check)
        if hasattr(svm_primal, 'losses') and hasattr(svm_primal, 'accuracies'):
            primal_losses = getattr(svm_primal, 'losses', [])
            primal_accuracies = getattr(svm_primal, 'accuracies', [])
            if primal_losses and primal_accuracies:
                 model_histories['Primal SVM (Hinge)'] = {'losses': primal_losses, 'accuracies': primal_accuracies}
                 model_names.append('Primal SVM (Hinge)')
                 print("Found history for Primal SVM (Hinge)")
            else:
                 print("WARN: 'svm_primal' object found, but history lists ('losses', 'accuracies') are empty.")
        else:
            print("WARN: 'svm_primal' object found, but lacks expected history attributes.")
    else:
        # This is the expected case if the variable wasn't renamed or doesn't exist
        print("INFO: 'svm_primal' object potentially not found or wasn't renamed. Skipping Primal SVM (Hinge).")
except NameError:
    # This handles cases where the variable truly doesn't exist
    print("INFO: 'svm_primal' variable does not exist. Skipping Primal SVM (Hinge).")


# 2. Dual SVM (RBF Kernel)
try:
    if 'dual_svm' in locals() or 'dual_svm' in globals():
        # Check if it's the correct type using unique attributes/methods
        is_dual_svm = hasattr(dual_svm, 'compute_kernel_matrix') and hasattr(dual_svm, 'support_vectors') and hasattr(dual_svm, 'rbf_kernel')
        if is_dual_svm:
             dual_losses = getattr(dual_svm, 'losses', [])
             dual_accuracies = getattr(dual_svm, 'accuracies', [])
             if dual_losses and dual_accuracies:
                  model_histories['Dual SVM (RBF)'] = {'losses': dual_losses, 'accuracies': dual_accuracies}
                  model_names.append('Dual SVM (RBF)')
                  print("Found history for Dual SVM (RBF)")
             else:
                  print("WARN: 'dual_svm' (Dual SVM) object found, but history lists ('losses', 'accuracies') are empty.")
        else:
             print("WARN: 'dual_svm' object found, but it doesn't seem to be the Dual SVM (missing expected attributes). Skipping.")
    else:
        print("INFO: 'dual_svm' object not found. Skipping Dual SVM (RBF).")
except NameError:
     print("INFO: 'dual_svm' variable does not exist. Skipping Dual SVM (RBF).")


# 3. squared_hinge_svm (Squared Hinge Loss)
try:
    if 'squared_hinge_svm' in locals() or 'squared_hinge_svm' in globals():
         # Check if it's the correct type using unique attributes/methods
         is_sq_hinge_svm = hasattr(squared_hinge_svm, '_squared_hinge_loss') and hasattr(squared_hinge_svm, 'history') and callable(getattr(squared_hinge_svm, 'predict', None))
         if is_sq_hinge_svm:
            sq_hinge_history = getattr(squared_hinge_svm, 'history', {})
            sq_hinge_losses = sq_hinge_history.get('train_loss', [])
            sq_hinge_accuracies = sq_hinge_history.get('train_acc', []) # Use train_acc for comparison
            if sq_hinge_losses and sq_hinge_accuracies:
                 # Convert accuracy from fraction to percentage if needed
                 sq_hinge_accuracies_pct = [a * 100 if a <= 1.0 else a for a in sq_hinge_accuracies]
                 model_histories['SVM (Squared Hinge)'] = {'losses': sq_hinge_losses, 'accuracies': sq_hinge_accuracies_pct}
                 model_names.append('SVM (Squared Hinge)')
                 print("Found history for SVM (Squared Hinge)")
            else:
                 print("WARN: 'svm' (Sq Hinge) object found, but history dict or its keys are empty.")
         else:
              # This check helps if 'svm' was overwritten by something unexpected
              print("WARN: 'svm' object found, but it doesn't seem to be the Squared Hinge SVM (missing expected attributes). Skipping.")
    else:
        print("INFO: 'svm' object not found. Skipping SVM (Squared Hinge).")
except NameError:
     print("INFO: 'svm' variable does not exist. Skipping SVM (Squared Hinge).")


# --- Plotting Convergence ---

if not model_histories:
    print("\nNo valid model histories found to plot convergence.")
elif len(model_histories) < 2:
    print(f"\nOnly found history for {len(model_histories)} model. Cannot plot comparison meaningfully.")
    # Optionally, plot the single history if desired
    # single_model_name = model_names[0]
    # plot_single_history(model_histories[single_model_name]) # You would need to define this function
else:
    print(f"\nPlotting convergence for: {', '.join(model_names)}")
    n_plots = len(model_histories)
    # Using specific colors instead of colormap for consistency
    color_map = {
        'Primal SVM (Hinge)': '#767497',
        'Dual SVM (RBF)': '#f7c3d4',
        'SVM (Squared Hinge)': '#aadd99'
    }
    default_color = '#808080' # Grey for models not in map

    fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=False) # sharex=False might be better if scales differ

    # Plot Losses
    axs[0].set_title('Training Loss vs. Epochs / Iterations')
    axs[0].set_ylabel('Loss')
    axs[0].grid(True, linestyle='--', alpha=0.6)
    # Consider log scale if losses vary widely:
    # try:
    #    axs[0].set_yscale('log')
    # except ValueError: # Handle potential non-positive values if using log scale
    #    print("Log scale for loss failed, using linear scale.")
    #    axs[0].set_yscale('linear')


    # Plot Accuracies
    axs[1].set_title('Training Accuracy vs. Epochs / Iterations')
    axs[1].set_xlabel('Epochs / Iterations (Note: Dual SVM sampled)')
    axs[1].set_ylabel('Accuracy (%)')
    axs[1].set_ylim(0, 105) # Accuracy typically 0-100
    axs[1].grid(True, linestyle='--', alpha=0.6)

    max_x_loss = 0
    max_x_acc = 0

    for name in model_names:
        history = model_histories[name]
        losses = history['losses']
        accuracies = history['accuracies']
        epochs = len(losses) # Number of recorded points
        current_color = color_map.get(name, default_color)

        # Determine x-axis values based on how history was recorded
        if name == 'Dual SVM (RBF)':
            # Assuming history was recorded every 100 epochs
            x_values = np.arange(1, epochs + 1) * 100
            plot_label = f'{name} (Sampled every 100 epochs)'
        else:
            # Primal and Squared Hinge record every iteration/epoch
            x_values = np.arange(1, epochs + 1)
            plot_label = name

        if epochs > 0: # Only plot if there's data
            # Plot loss
            axs[0].plot(x_values, losses, label=plot_label, color=current_color, alpha=0.8, linewidth=1.5)
            max_x_loss = max(max_x_loss, x_values[-1])

            # Plot accuracy
            axs[1].plot(x_values, accuracies, label=plot_label, color=current_color, alpha=0.8, linewidth=1.5)
            max_x_acc = max(max_x_acc, x_values[-1])
        else:
            print(f"Skipping plot for {name} due to empty history.")

    # Set reasonable x-limits based on the longest run, only if plots were made
    if max_x_loss > 0:
        axs[0].set_xlim(0, max_x_loss * 1.05)
    if max_x_acc > 0:
        axs[1].set_xlim(0, max_x_acc * 1.05)

    axs[0].legend()
    axs[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_models(models, X_test, y_test, model_names=None):

    if model_names is None:
        model_names = [f"Model {i+1}" for i in range(len(models))]

    y_true = np.where(y_test <= 0, 0, 1)
    results = {}

    for model, name in zip(models, model_names):
        start_time = time.time()
        y_pred = model.predict(X_test)
        y_pred_binary = np.where(y_pred <= 0, 0, 1)
        end_time = time.time()

        # Handle loss curve
        loss_curve = getattr(model, 'losses', [])
        if not loss_curve:
            loss_curve = model.history.get('val_loss', [])

        # Handle accuracy curve, ensuring it's scaled to percentage
        acc_curve = getattr(model, 'accuracies', [])
        if not acc_curve:
            acc_curve = [acc * 100 for acc in model.history.get('val_acc', [])]

        results[name] = {
            "accuracy": accuracy_score(y_true, y_pred_binary) * 100,
            "precision": precision_score(y_true, y_pred_binary, zero_division=0) * 100,
            "recall": recall_score(y_true, y_pred_binary, zero_division=0) * 100,
            "f1_score": f1_score(y_true, y_pred_binary, zero_division=0) * 100,
            "confusion_matrix": confusion_matrix(y_true, y_pred_binary),
            "inference_time": end_time - start_time,
            "loss_curve": loss_curve,
            "acc_curve": acc_curve
        }

    return results


In [ ]:
results = evaluate_models(
    models=[svm_primal, dual_svm, squared_hinge_svm],
    X_test=X_test,
    y_test=y_test,
    model_names=["Primal SVM", "Dual RBF SVM", "Squared Hinge SVM"]
)


In [ ]:
for name, metrics in results.items():
    print(f"\n{name} Evaluation:")
    print(f"Accuracy : {metrics['accuracy']:.2f}%")
    print(f"Precision: {metrics['precision']:.2f}%")
    print(f"Recall   : {metrics['recall']:.2f}%")
    print(f"F1 Score : {metrics['f1_score']:.2f}%")
    print(f"Inference Time: {metrics['inference_time']:.4f}s")


In [ ]:
model_names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in model_names]
f1_scores = [results[name]['f1_score'] for name in model_names]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=model_names,
    y=accuracies,
    name="Accuracy",
    marker_color="#767497"
))
fig.add_trace(go.Bar(
    x=model_names,
    y=f1_scores,
    name="F1 Score",
    marker_color="#f7c3d4"
))

fig.update_layout(
    barmode="group",
    title="Model Accuracy and F1 Score",
    yaxis_title="Score (%)",
    height=400
)
fig.show()


In [ ]:
# Custom pink-lavender colormap
custom_colorscale = [
    [0.0, "#666573"],
    [1.0, "#f7c3d4"]
]

# Create subplot layout for 3 models
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=model_names  # only model names as titles
)

for i, name in enumerate(model_names):
    cm = results[name]["confusion_matrix"]

    heatmap = go.Heatmap(
        z=cm,
        x=["Pred 0", "Pred 1"],
        y=["True 0", "True 1"],
        colorscale=custom_colorscale,
        showscale=False,
        text=[[str(cell) for cell in row] for row in cm],
        texttemplate="%{text}",
        textfont={"color": "white", "size": 18},
        hovertemplate=" %{y},%{x}: %{z}<extra></extra>"
    )

    fig.add_trace(heatmap, row=1, col=i + 1)

fig.update_layout(
    height=400,
    width=1000,
    margin=dict(t=30, b=20),
    plot_bgcolor='white'
)

fig.show()


In [ ]:
def plot_inference_times(results):
    model_names = list(results.keys())
    inference_times = [results[name]['inference_time'] for name in model_names]

    fig = go.Figure(data=[
        go.Bar(
            x=model_names,
            y=inference_times,
            text=[f"{t:.4f} s" for t in inference_times],
            textposition='auto',
            marker_color='teal'
        )
    ])

    fig.update_layout(
        title="Inference Time Comparison",
        xaxis_title="Model",
        yaxis_title="Time (seconds)",
        template="plotly_white"
    )

    fig.show()
plot_inference_times(results)